In [1]:
# Locate the repository when running from any notebook subfolder.
from pathlib import Path
import os
_notebook_start = Path.cwd().resolve()
_notebook_root = next(
    (p for p in [_notebook_start, *_notebook_start.parents]
     if (p / "docs" / "predictor_list.csv").is_file()
     and (p / "notebooks").is_dir()), None
)
if _notebook_root is None:
    raise FileNotFoundError("Cannot locate the project repository.")
os.chdir(_notebook_root)


# Logistic Regression for Late Delivery

## Objective

Examine whether distance, freight, holidays and
pre-purchase rainfall remain associated with late
delivery after adjustment for other selected factors.

The analysis uses the shared UTC dataset and the
training orders defined in the split manifest.

M0 and M4 are the primary specifications.
Results will be reported using adjusted odds ratios,
95% confidence intervals and model diagnostics.

The analysis is observational and does not establish
causation.

In [2]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import sys

import numpy as np
import pandas as pd
import statsmodels.api as sm

current_dir = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_dir, *current_dir.parents]
        if (path / "docs" / "predictor_list.csv").is_file()
        and (
            path
            / "data" / "ml_preparation"
            / "split_manifest.csv"
        ).is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the Stage 4 repository."
    )

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "research_data_weather_price_holiday_assume_utc.csv"
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "data" / "ml_preparation"
    / "split_manifest.csv"
)

SUMMARY_PATH = MANIFEST_PATH.with_name("split_summary.csv")
PREDICTOR_PATH = PROJECT_ROOT / "docs" / "predictor_list.csv"

print("Project root:", PROJECT_ROOT)
print("Python:", sys.executable)

for package in ["pandas", "numpy", "scipy", "statsmodels", "patsy"]:
    print(f"{package}: {version(package)}")

Project root: C:\Users\LENOVO\OneDrive\Desktop\PTTK
Python: C:\Users\LENOVO\OneDrive\Desktop\PTTK\.venv-boosting\Scripts\python.exe
pandas: 2.3.3
numpy: 2.3.5
scipy: 1.16.3
statsmodels: 0.15.0
patsy: 1.0.3


In [3]:
# Read shared inputs without modifying source files.

data = pd.read_csv(
    DATA_PATH,
    dtype={"order_id": "string"},
    low_memory=False,
)

manifest = pd.read_csv(
    MANIFEST_PATH,
    dtype={
        "order_id": "string",
        "split": "string",
    },
)

split_summary = pd.read_csv(SUMMARY_PATH)
predictor_list = pd.read_csv(
    PREDICTOR_PATH,
    dtype="string",
)

assert data["order_id"].notna().all()
assert data["order_id"].is_unique

assert manifest["order_id"].notna().all()
assert manifest["order_id"].is_unique
assert manifest["order_id"].isin(data["order_id"]).all()

assert predictor_list["feature"].notna().all()
assert predictor_list["feature"].is_unique
assert len(predictor_list) == 21

# Select train IDs before preparing any model inputs.
train_manifest = manifest.loc[
    manifest["split"].eq("train"),
    ["order_id", "purchase_time", "exclusion_reason"],
].copy()

assert (
    train_manifest["exclusion_reason"]
    .fillna("")
    .str.strip()
    .eq("")
).all()

train_raw = train_manifest.merge(
    data,
    on="order_id",
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert train_raw["_merge"].eq("both").all()
train_raw = train_raw.drop(columns="_merge")

expected = split_summary.loc[
    split_summary["final_split"].eq("train")
]

assert len(expected) == 1

expected_n = int(expected.iloc[0]["orders"])
expected_late = int(expected.iloc[0]["late_orders"])

assert len(train_raw) == expected_n == 57449
assert train_raw["is_late"].isin([0, 1]).all()
assert train_raw["is_late"].eq(1).sum() == expected_late == 3720

# UTC is the agreed working assumption.
manifest_purchase = pd.to_datetime(
    train_raw["purchase_time"],
    format="ISO8601",
    errors="raise",
    utc=True,
)

source_purchase = pd.to_datetime(
    train_raw["order_purchase_timestamp"],
    format="ISO8601",
    errors="raise",
    utc=True,
)

delivery_time = pd.to_datetime(
    train_raw["order_delivered_customer_date"],
    format="ISO8601",
    errors="raise",
    utc=True,
)

assert manifest_purchase.notna().all()
assert source_purchase.notna().all()
assert manifest_purchase.eq(source_purchase).all()

assert source_purchase.lt(
    pd.Timestamp("2018-03-02", tz="UTC")
).all()

# Delivery dates are used only to audit label availability.
assert delivery_time.notna().all()
assert delivery_time.lt(
    pd.Timestamp("2018-04-16", tz="UTC")
).all()

# Main specifications follow the documented Stage 4 roles.
main_predictors = predictor_list.loc[
    predictor_list["stage4_role"].isin(
        ["Primary", "Adjustment", "Explanatory"]
    )
].copy()

main_features = main_predictors["feature"].tolist()

basic_features = main_predictors.loc[
    main_predictors["group"].eq("F0"),
    "feature",
].tolist()

assert len(basic_features) == 6
assert len(main_features) == 13
assert set(main_features).issubset(train_raw.columns)

# Inspect missingness before selecting an analysis sample.
missing_summary = pd.DataFrame({
    "feature": main_features,
    "dtype": [
        str(train_raw[column].dtype)
        for column in main_features
    ],
    "missing_orders": [
        int(train_raw[column].isna().sum())
        for column in main_features
    ],
    "unique_nonmissing": [
        train_raw[column].nunique()
        for column in main_features
    ],
})

missing_summary["missing_percent"] = (
    missing_summary["missing_orders"] / len(train_raw) * 100
)

complete_for_main = (
    train_raw[main_features].notna().all(axis=1)
)

with DATA_PATH.open("rb") as stream:
    dataset_sha256 = hashlib.file_digest(
        stream, "sha256"
    ).hexdigest()

print("TRAIN SAMPLE CHECK")
print("Orders:", len(train_raw))
print("Late:", int(train_raw["is_late"].eq(1).sum()))
print("Not late:", int(train_raw["is_late"].eq(0).sum()))
print("Dataset SHA256:", dataset_sha256)

print("\nMAIN PREDICTOR MISSINGNESS")
display(missing_summary.round({"missing_percent": 3}))

print("\nCOMPLETE-CASE CANDIDATE FOR M0 AND M4")
print("Orders:", int(complete_for_main.sum()))
print(
    "Late:",
    int(
        train_raw.loc[
            complete_for_main, "is_late"
        ].eq(1).sum()
    ),
)
print(
    "Orders with at least one missing main predictor:",
    int((~complete_for_main).sum()),
)

print("\nCATEGORY VALUES")
for column in [
    "purchase_weekday",
    "purchase_is_national_holiday",
]:
    print(f"\n{column}")
    print(
        train_raw[column]
        .value_counts(dropna=False)
        .to_string()
    )

print("\nPASS: Shared training sample and cutoffs match.")

TRAIN SAMPLE CHECK
Orders: 57449
Late: 3720
Not late: 53729
Dataset SHA256: ed604fa22a2c95695036966f80a0f8a4666fb242bc2ab9f781ed8fb68630ff4c

MAIN PREDICTOR MISSINGNESS


,feature,dtype,missing_orders,unique_nonmissing,missing_percent
0,distance_km_mean,float64,300,54358,0.522
1,customer_state,object,0,27,0.000
2,item_count,float64,0,16,0.000
3,seller_count,float64,0,5,0.000
4,purchase_weekday,object,0,7,0.000
5,estimated_window_days,int64,0,94,0.000
6,items_value,float64,0,5300,0.000
7,freight_value,float64,0,5644,0.000
8,total_product_weight_g,float64,16,2177,0.028
9,total_product_volume_cm3,float64,16,4689,0.028



COMPLETE-CASE CANDIDATE FOR M0 AND M4
Orders: 56553
Late: 3649
Orders with at least one missing main predictor: 896

CATEGORY VALUES

purchase_weekday
purchase_weekday
Tuesday      9254
Monday       9150
Wednesday    8865
Thursday     8530
Friday       8420
Sunday       6859
Saturday     6371

purchase_is_national_holiday
purchase_is_national_holiday
0    56541
1      908

PASS: Shared training sample and cutoffs match.


## 2. Analysis Sample and Predictor Distributions

M0 and M4 use the same complete-case training sample,
defined using the 13 predictors in the main specification.

Orders with missing predictors are counted separately.
Their exclusion may affect the analysis if missingness
is associated with delivery outcomes.

Predictor ranges and outcome counts within categories
are inspected before fitting the models.

In [4]:
# Use the same complete-case sample for both main models.
train_analysis = train_raw.loc[complete_for_main].copy()

sample_rows = []

for name, sample in [
    ("All training orders", train_raw),
    ("Complete-case sample", train_analysis),
    ("Excluded for missing predictors", train_raw.loc[~complete_for_main]),
]:
    n_orders = len(sample)
    n_late = int(sample["is_late"].eq(1).sum())

    sample_rows.append({
        "sample": name,
        "orders": n_orders,
        "not_late_orders": int(sample["is_late"].eq(0).sum()),
        "late_orders": n_late,
        "late_percent": n_late / n_orders * 100 if n_orders else np.nan,
    })

sample_audit = pd.DataFrame(sample_rows)

assert len(train_analysis) == 56553
assert train_analysis["is_late"].eq(1).sum() == 3649
assert train_analysis[main_features].notna().all().all()

print("ANALYSIS SAMPLE")
display(sample_audit.round(3))


def outcome_counts(frame, column):
    counts = (
        frame.groupby(column, observed=True)["is_late"]
        .agg(orders="size", late_orders="sum")
        .reset_index()
    )

    counts["late_orders"] = counts["late_orders"].astype(int)
    counts["not_late_orders"] = (
        counts["orders"] - counts["late_orders"]
    )
    counts["late_percent"] = (
        counts["late_orders"] / counts["orders"] * 100
    )
    counts["single_outcome_group"] = (
        counts["late_orders"].eq(0)
        | counts["not_late_orders"].eq(0)
    )

    return counts


print("\nCUSTOMER STATE")
display(
    outcome_counts(train_analysis, "customer_state")
    .sort_values("orders")
    .round(3)
)

print("\nSELLER COUNT")
display(outcome_counts(train_analysis, "seller_count").round(3))

print("\nPURCHASE HOLIDAY")
display(
    outcome_counts(train_analysis, "purchase_is_national_holiday")
    .round(3)
)


numeric_features = [
    feature for feature in main_features
    if feature not in ["customer_state", "purchase_weekday"]
]

numeric_data = train_analysis[numeric_features].apply(
    pd.to_numeric,
    errors="raise",
)

assert np.isfinite(numeric_data.to_numpy(dtype=float)).all()

distribution_summary = (
    numeric_data.describe(
        percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
    )
    .T[
        ["min", "1%", "25%", "50%", "75%", "99%", "max"]
    ]
)

print("\nNUMERIC PREDICTOR DISTRIBUTIONS")
display(distribution_summary.round(3))

print("\nPASS: Complete-case sample contains no missing or nonfinite predictors.")

ANALYSIS SAMPLE


,sample,orders,not_late_orders,late_orders,late_percent
0,All training orders,57449,53729,3720,6.475
1,Complete-case sample,56553,52904,3649,6.452
2,Excluded for missing predictors,896,825,71,7.924



CUSTOMER STATE


,customer_state,orders,late_orders,not_late_orders,late_percent,single_outcome_group
21,RR,10,1,9,10.000,False
3,AP,27,2,25,7.407,False
0,AC,28,2,26,7.143,False
2,AM,68,1,67,1.471,False
20,RO,107,1,106,0.935,False
26,TO,165,14,151,8.485,False
24,SE,219,30,189,13.699,False
1,AL,238,55,183,23.109,False
14,PB,270,34,236,12.593,False
19,RN,286,24,262,8.392,False



SELLER COUNT


,seller_count,orders,late_orders,not_late_orders,late_percent,single_outcome_group
0,1.0,55925,3641,52284,6.511,False
1,2.0,599,8,591,1.336,False
2,3.0,26,0,26,0.000,True
3,4.0,2,0,2,0.000,True
4,5.0,1,0,1,0.000,True



PURCHASE HOLIDAY


,purchase_is_national_holiday,orders,late_orders,not_late_orders,late_percent,single_outcome_group
0,0,55667,3613,52054,6.490,False
1,1,886,36,850,4.063,False



NUMERIC PREDICTOR DISTRIBUTIONS


,min,1%,25%,50%,75%,99%,max
distance_km_mean,0.00,6.087,222.904,448.310,807.973,2476.279,3387.259
item_count,1.00,1.000,1.000,1.000,1.000,3.000,21.000
seller_count,1.00,1.000,1.000,1.000,1.000,2.000,5.000
estimated_window_days,8.00,11.000,21.000,25.000,30.000,52.000,156.000
items_value,2.29,11.900,45.900,84.990,149.900,962.875,13440.000
freight_value,0.00,7.780,14.090,16.460,23.360,95.387,1002.290
total_product_weight_g,50.00,100.000,300.000,800.000,2150.000,22468.800,184400.000
total_product_volume_cm3,288.00,352.000,3072.000,7942.000,20539.000,141192.000,1476000.000
purchase_is_national_holiday,0.00,0.000,0.000,0.000,0.000,1.000,1.000
national_holiday_count_in_estimated_window,0.00,0.000,0.000,0.000,1.000,3.000,5.000



PASS: Complete-case sample contains no missing or nonfinite predictors.


## 3. Initial Baseline Model

M0 includes distance, customer state, item count,
seller multiplicity, purchase weekday and the estimated
delivery window.

Distance is expressed in units of 100 km.
Customer state uses SP as the reference category.
Purchase weekday uses Monday as the reference category.

Seller count is represented by a binary indicator:
one seller versus two or more sellers. Separate categories
for three or more sellers contain few observations and
no late orders in the analysis sample.

This initial specification assumes linear associations
with the log odds for numeric predictors. Functional form
and other model diagnostics will be examined before
reporting the final results.

The model is fitted without class weights, resampling
or regularization.

In [5]:
import warnings

from patsy import dmatrices

model_data = train_analysis.copy()

model_data["is_late"] = model_data["is_late"].astype(int)
model_data["distance_100km"] = (
    model_data["distance_km_mean"] / 100
)
model_data["is_multi_seller"] = (
    model_data["seller_count"] >= 2
).astype(int)

# Use ordinary string objects for categorical formula terms.
for column in ["customer_state", "purchase_weekday"]:
    model_data[column] = model_data[column].astype(object)

assert "SP" in model_data["customer_state"].unique()
assert "Monday" in model_data["purchase_weekday"].unique()
assert model_data["seller_count"].ge(1).all()

m0_formula = (
    'is_late ~ distance_100km'
    ' + C(customer_state, Treatment(reference="SP"))'
    ' + item_count'
    ' + is_multi_seller'
    ' + C(purchase_weekday, Treatment(reference="Monday"))'
    ' + estimated_window_days'
)

y_m0, X_m0 = dmatrices(
    m0_formula,
    data=model_data,
    return_type="dataframe",
    NA_action="raise",
)

assert len(X_m0) == len(model_data)
assert X_m0.index.equals(model_data.index)
assert np.isfinite(X_m0.to_numpy()).all()

design_rank = np.linalg.matrix_rank(X_m0.to_numpy())

print("M0 DESIGN CHECK")
print("Orders:", len(X_m0))
print("Parameters including intercept:", X_m0.shape[1])
print("Design matrix rank:", design_rank)

assert design_rank == X_m0.shape[1], (
    "The design matrix contains exact linear dependencies."
)

print("\nSELLER MULTIPLICITY")
display(
    outcome_counts(model_data, "is_multi_seller").round(3)
)

M0 DESIGN CHECK
Orders: 56553
Parameters including intercept: 37
Design matrix rank: 37

SELLER MULTIPLICITY


,is_multi_seller,orders,late_orders,not_late_orders,late_percent,single_outcome_group
0,0,55925,3641,52284,6.511,False
1,1,628,8,620,1.274,False


In [6]:
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")

    m0_initial = sm.GLM(
        y_m0.iloc[:, 0],
        X_m0,
        family=sm.families.Binomial(
            link=sm.families.links.Logit()
        ),
        missing="raise",
    ).fit(
        maxiter=100,
        tol=1e-8,
    )

m0_fitted = np.asarray(m0_initial.fittedvalues)

m0_initial_checks = pd.DataFrame([{
    "model": "M0_initial",
    "orders": int(m0_initial.nobs),
    "parameters": len(m0_initial.params),
    "design_rank": design_rank,
    "converged": bool(m0_initial.converged),
    "iterations": m0_initial.fit_history.get("iteration"),
    "finite_coefficients": bool(
        np.isfinite(m0_initial.params).all()
    ),
    "finite_standard_errors": bool(
        np.isfinite(m0_initial.bse).all()
    ),
    "minimum_fitted_probability": m0_fitted.min(),
    "maximum_fitted_probability": m0_fitted.max(),
    "warning_count": len(caught_warnings),
}])

print("M0 INITIAL FIT CHECK")
display(m0_initial_checks.T)

print("\nFIT WARNINGS")
if caught_warnings:
    for warning in caught_warnings:
        print(
            f"{warning.category.__name__}: "
            f"{warning.message}"
        )
else:
    print("No warnings were emitted.")

initial_coefficients = pd.DataFrame({
    "coefficient": m0_initial.params,
    "standard_error_initial": m0_initial.bse,
})

print("\nMAIN NUMERIC AND SELLER TERMS")
display(
    initial_coefficients.loc[
        [
            "distance_100km",
            "item_count",
            "is_multi_seller",
            "estimated_window_days",
        ]
    ].round(6)
)

print("\nEIGHT TERMS WITH THE LARGEST INITIAL STANDARD ERRORS")
display(
    initial_coefficients
    .sort_values("standard_error_initial", ascending=False)
    .head(8)
    .round(6)
)

M0 INITIAL FIT CHECK


,0
model,M0_initial
orders,56553
parameters,37
design_rank,37
converged,True
iterations,7
finite_coefficients,True
finite_standard_errors,True
minimum_fitted_probability,0.000049
maximum_fitted_probability,0.449441



FIT WARNINGS
No warnings were emitted.

MAIN NUMERIC AND SELLER TERMS


,coefficient,standard_error_initial
distance_100km,0.055717,0.006184
item_count,0.000423,0.034277
is_multi_seller,-1.628638,0.359967
estimated_window_days,-0.056912,0.003494



EIGHT TERMS WITH THE LARGEST INITIAL STANDARD ERRORS


,coefficient,standard_error_initial
"C(customer_state, Treatment(reference=""SP""))[T.RR]",0.507837,1.073290
"C(customer_state, Treatment(reference=""SP""))[T.AM]",-1.269104,1.018978
"C(customer_state, Treatment(reference=""SP""))[T.RO]",-1.516908,1.012543
"C(customer_state, Treatment(reference=""SP""))[T.AP]",0.919224,0.754622
"C(customer_state, Treatment(reference=""SP""))[T.AC]",0.635904,0.751414
is_multi_seller,-1.628638,0.359967
"C(customer_state, Treatment(reference=""SP""))[T.TO]",0.631939,0.292061
"C(customer_state, Treatment(reference=""SP""))[T.MS]",0.188044,0.269949


## 4. Initial Full Model

M4 extends M0 with item value, freight, product weight,
product volume, holiday indicators and destination
rainfall before purchase.

Both models use the same complete-case training sample.

Numeric predictors are rescaled to interpretable units.
Rescaling does not change the assumed functional form.

Weather represents a state-level index rather than
rainfall measured at the customer's exact location.

This initial model is checked for numerical stability
before examining multicollinearity, functional form
and the uncertainty of the final estimates.

In [7]:
# Rescale predictors without changing the source columns.
model_data["items_value_100brl"] = (
    model_data["items_value"] / 100
)
model_data["freight_value_10brl"] = (
    model_data["freight_value"] / 10
)
model_data["product_weight_kg"] = (
    model_data["total_product_weight_g"] / 1000
)
model_data["product_volume_10l"] = (
    model_data["total_product_volume_cm3"] / 10000
)
model_data["destination_rain_5mm"] = (
    model_data["destination_rain_state_mean_mm"] / 5
)

m4_added_terms = [
    "items_value_100brl",
    "freight_value_10brl",
    "product_weight_kg",
    "product_volume_10l",
    "purchase_is_national_holiday",
    "national_holiday_count_in_estimated_window",
    "destination_rain_5mm",
]

m4_formula = m0_formula + " + " + " + ".join(m4_added_terms)

y_m4, X_m4 = dmatrices(
    m4_formula,
    data=model_data,
    return_type="dataframe",
    NA_action="raise",
)

assert X_m4.index.equals(X_m0.index)
assert y_m4.equals(y_m0)
assert np.isfinite(X_m4.to_numpy()).all()

m4_design_rank = np.linalg.matrix_rank(X_m4.to_numpy())

print("M4 DESIGN CHECK")
print("Orders:", len(X_m4))
print("Parameters including intercept:", X_m4.shape[1])
print("Design matrix rank:", m4_design_rank)
print("Same orders and outcomes as M0:", y_m4.equals(y_m0))

assert m4_design_rank == X_m4.shape[1], (
    "The design matrix contains exact linear dependencies."
)

M4 DESIGN CHECK
Orders: 56553
Parameters including intercept: 44
Design matrix rank: 44
Same orders and outcomes as M0: True


In [8]:
with warnings.catch_warnings(record=True) as m4_warnings:
    warnings.simplefilter("always")

    m4_initial = sm.GLM(
        y_m4.iloc[:, 0],
        X_m4,
        family=sm.families.Binomial(
            link=sm.families.links.Logit()
        ),
        missing="raise",
    ).fit(
        maxiter=100,
        tol=1e-8,
    )

m4_fitted = np.asarray(m4_initial.fittedvalues)

m4_initial_checks = pd.DataFrame([{
    "model": "M4_initial",
    "orders": int(m4_initial.nobs),
    "parameters": len(m4_initial.params),
    "design_rank": m4_design_rank,
    "converged": bool(m4_initial.converged),
    "iterations": m4_initial.fit_history.get("iteration"),
    "finite_coefficients": bool(
        np.isfinite(m4_initial.params).all()
    ),
    "finite_standard_errors": bool(
        np.isfinite(m4_initial.bse).all()
    ),
    "minimum_fitted_probability": m4_fitted.min(),
    "maximum_fitted_probability": m4_fitted.max(),
    "warning_count": len(m4_warnings),
}])

print("INITIAL FIT CHECKS")
display(
    pd.concat(
        [m0_initial_checks, m4_initial_checks],
        ignore_index=True,
    ).set_index("model").T
)

print("\nM4 FIT WARNINGS")
if m4_warnings:
    for warning in m4_warnings:
        print(
            f"{warning.category.__name__}: "
            f"{warning.message}"
        )
else:
    print("No warnings were emitted.")

m4_initial_coefficients = pd.DataFrame({
    "coefficient": m4_initial.params,
    "standard_error_initial": m4_initial.bse,
})

print("\nADDED TERMS IN M4")
display(
    m4_initial_coefficients.loc[m4_added_terms].round(6)
)

print("\nEIGHT TERMS WITH THE LARGEST INITIAL STANDARD ERRORS")
display(
    m4_initial_coefficients
    .sort_values("standard_error_initial", ascending=False)
    .head(8)
    .round(6)
)

INITIAL FIT CHECKS


model,M0_initial,M4_initial
orders,56553,56553
parameters,37,44
design_rank,37,44
converged,True,True
iterations,7,7
finite_coefficients,True,True
finite_standard_errors,True,True
minimum_fitted_probability,0.000049,0.000035
maximum_fitted_probability,0.449441,0.676023
warning_count,0,0



M4 FIT WARNINGS
No warnings were emitted.

ADDED TERMS IN M4


,coefficient,standard_error_initial
items_value_100brl,0.016116,0.007375
freight_value_10brl,-0.000094,0.013417
product_weight_kg,0.016636,0.005995
product_volume_10l,0.011858,0.009405
purchase_is_national_holiday,-0.529789,0.173871
national_holiday_count_in_estimated_window,-0.036240,0.024567
destination_rain_5mm,0.094595,0.011367



EIGHT TERMS WITH THE LARGEST INITIAL STANDARD ERRORS


,coefficient,standard_error_initial
"C(customer_state, Treatment(reference=""SP""))[T.RR]",0.468021,1.081677
"C(customer_state, Treatment(reference=""SP""))[T.AM]",-1.274676,1.019346
"C(customer_state, Treatment(reference=""SP""))[T.RO]",-1.511393,1.012704
"C(customer_state, Treatment(reference=""SP""))[T.AP]",0.923397,0.759307
"C(customer_state, Treatment(reference=""SP""))[T.AC]",0.660170,0.752103
is_multi_seller,-1.559043,0.360140
"C(customer_state, Treatment(reference=""SP""))[T.TO]",0.615190,0.293312
"C(customer_state, Treatment(reference=""SP""))[T.MS]",0.188816,0.270677


## 5. Model Diagnostics

### 5.1. Multicollinearity

Variance inflation factors are calculated for numeric
and binary predictor terms in M0 and M4.

Each calculation includes all other columns in the
corresponding design matrix, including state and
weekday indicators.

VIF values above 5 are flagged for review.
Predictors are not removed automatically.

Pairwise Pearson correlations provide an additional
description of relationships among the numeric and
binary predictors. These correlations do not replace
the VIF assessment.

In [9]:
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor,
)


def numeric_term_vif(model_name, design_matrix):
    matrix = design_matrix.to_numpy(dtype=float)
    rows = []

    for position, term in enumerate(design_matrix.columns):
        # Category indicators remain in the auxiliary regressions.
        if term == "Intercept" or term.startswith("C("):
            continue

        vif = variance_inflation_factor(matrix, position)

        rows.append({
            "model": model_name,
            "term": term,
            "vif": vif,
            "flag_vif_above_5": vif > 5,
        })

    return pd.DataFrame(rows)


vif_m0 = numeric_term_vif("M0_initial", X_m0)
vif_m4 = numeric_term_vif("M4_initial", X_m4)

vif_results = pd.concat(
    [vif_m0, vif_m4],
    ignore_index=True,
)

print("M0 VIF")
display(
    vif_m0.sort_values("vif", ascending=False).round(3)
)

print("\nM4 VIF")
display(
    vif_m4.sort_values("vif", ascending=False).round(3)
)

print("\nTERMS FLAGGED FOR REVIEW")
flagged_vif = vif_results.loc[
    vif_results["flag_vif_above_5"]
]

if flagged_vif.empty:
    print("No numeric or binary terms have VIF above 5.")
else:
    display(flagged_vif.round(3))

M0 VIF


,model,term,vif,flag_vif_above_5
0,M0_initial,distance_100km,4.814,False
3,M0_initial,estimated_window_days,1.536,False
1,M0_initial,item_count,1.065,False
2,M0_initial,is_multi_seller,1.065,False



M4 VIF


,model,term,vif,flag_vif_above_5
0,M4_initial,distance_100km,5.103,True
6,M4_initial,product_weight_kg,3.569,False
5,M4_initial,freight_value_10brl,3.478,False
7,M4_initial,product_volume_10l,3.392,False
3,M4_initial,estimated_window_days,1.666,False
1,M4_initial,item_count,1.487,False
4,M4_initial,items_value_100brl,1.237,False
9,M4_initial,national_holiday_count_in_estimated_window,1.095,False
2,M4_initial,is_multi_seller,1.066,False
10,M4_initial,destination_rain_5mm,1.023,False



TERMS FLAGGED FOR REVIEW


,model,term,vif,flag_vif_above_5
4,M4_initial,distance_100km,5.103,True


In [10]:
numeric_model_terms = [
    term for term in X_m4.columns
    if term != "Intercept" and not term.startswith("C(")
]

predictor_correlations = X_m4[numeric_model_terms].corr()

row_positions, column_positions = np.triu_indices(
    len(numeric_model_terms),
    k=1,
)

correlation_pairs = pd.DataFrame({
    "term_1": [
        numeric_model_terms[i] for i in row_positions
    ],
    "term_2": [
        numeric_model_terms[j] for j in column_positions
    ],
    "pearson_r": predictor_correlations.to_numpy()[
        row_positions, column_positions
    ],
})

correlation_pairs["absolute_r"] = (
    correlation_pairs["pearson_r"].abs()
)

print("TEN LARGEST ABSOLUTE PAIRWISE CORRELATIONS")
display(
    correlation_pairs
    .sort_values("absolute_r", ascending=False)
    .head(10)
    .round(3)
)

TEN LARGEST ABSOLUTE PAIRWISE CORRELATIONS


,term_1,term_2,pearson_r,absolute_r
45,product_weight_kg,product_volume_10l,0.821,0.821
40,freight_value_10brl,product_weight_kg,0.669,0.669
41,freight_value_10brl,product_volume_10l,0.664,0.664
2,distance_100km,estimated_window_days,0.545,0.545
13,item_count,freight_value_10brl,0.474,0.474
34,items_value_100brl,freight_value_10brl,0.422,0.422
35,items_value_100brl,product_weight_kg,0.345,0.345
4,distance_100km,freight_value_10brl,0.309,0.309
36,items_value_100brl,product_volume_10l,0.307,0.307
32,estimated_window_days,national_holiday_count_in_estimated_window,0.285,0.285


### 5.2. Functional Form Screening

Binned residual plots are used to inspect the initial
M4 specification.

Continuous predictors are divided into up to ten
quantile groups. Item count is grouped separately
because most orders contain one item.

For each group, the observed late-delivery rate is
compared with the mean fitted probability.

Systematic residual patterns may indicate an inadequate
functional form or other model misspecification.
Small residuals alone do not establish model adequacy.

This screening does not remove observations or change
the model specification.

In [11]:
import matplotlib.pyplot as plt

screening_terms = {
    "distance_100km": "Distance (100 km)",
    "estimated_window_days": "Estimated delivery window (days)",
    "items_value_100brl": "Item value (100 BRL)",
    "freight_value_10brl": "Freight (10 BRL)",
    "product_weight_kg": "Product weight (kg)",
    "product_volume_10l": "Product volume (10 litres)",
    "destination_rain_5mm": "Destination rainfall index (5 mm)",
    "item_count": "Item count",
}

assert X_m4.index.equals(model_data.index)

residual_tables = []

for term in screening_terms:
    check_data = pd.DataFrame({
        "value": model_data[term].to_numpy(dtype=float),
        "is_late": y_m4.iloc[:, 0].to_numpy(),
        "fitted_probability": m4_fitted,
    })

    if term == "item_count":
        check_data["band"] = pd.cut(
            check_data["value"],
            bins=[0, 1, 2, 3, np.inf],
            labels=["1", "2", "3", "4+"],
        )
    else:
        check_data["band"] = pd.qcut(
            check_data["value"],
            q=10,
            duplicates="drop",
        )

    assert check_data["band"].notna().all()

    grouped = (
        check_data.groupby("band", observed=True)
        .agg(
            orders=("is_late", "size"),
            late_orders=("is_late", "sum"),
            median_value=("value", "median"),
            observed_rate=("is_late", "mean"),
            fitted_rate=("fitted_probability", "mean"),
        )
        .reset_index()
    )

    grouped["residual_pp"] = (
        grouped["observed_rate"] - grouped["fitted_rate"]
    ) * 100

    grouped["term"] = term
    grouped["band"] = grouped["band"].astype(str)

    assert grouped["orders"].sum() == len(model_data)
    residual_tables.append(grouped)

binned_residuals_m4 = pd.concat(
    residual_tables,
    ignore_index=True,
)

screening_summary = (
    binned_residuals_m4
    .assign(
        absolute_residual_pp=lambda table:
        table["residual_pp"].abs()
    )
    .groupby("term", sort=False)
    .agg(
        groups=("orders", "size"),
        smallest_group_orders=("orders", "min"),
        largest_absolute_residual_pp=(
            "absolute_residual_pp", "max"
        ),
    )
    .reset_index()
)

print("M4 BINNED RESIDUAL SCREENING")
display(screening_summary.round(3))


fig_functional_form, axes = plt.subplots(
    4, 2,
    figsize=(15, 17),
    constrained_layout=True,
)

for ax, (term, title) in zip(
    axes.flat, screening_terms.items()
):
    plot_data = binned_residuals_m4.loc[
        binned_residuals_m4["term"].eq(term)
    ]

    positions = np.arange(len(plot_data))

    if term == "item_count":
        labels = plot_data["band"].tolist()
    else:
        labels = [
            f"{value:.2g}"
            for value in plot_data["median_value"]
        ]

    ax.plot(
        positions,
        plot_data["residual_pp"],
        marker="o",
    )
    ax.axhline(0, color="gray", linestyle="--")

    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=45)
    ax.set_title(title)
    ax.set_xlabel(
        "Item-count group" if term == "item_count"
        else "Median value within quantile group"
    )
    ax.set_ylabel("Mean residual (percentage points)")
    ax.grid(axis="y", alpha=0.25)

fig_functional_form.suptitle(
    "Initial M4: binned residual screening",
    fontsize=15,
)

plt.show()

M4 BINNED RESIDUAL SCREENING


,term,groups,smallest_group_orders,largest_absolute_residual_pp
0,distance_100km,10,5654,1.675
1,estimated_window_days,10,3159,1.589
2,items_value_100brl,10,5586,1.157
3,freight_value_10brl,10,5167,1.717
4,product_weight_kg,10,2482,0.957
5,product_volume_10l,10,5547,1.015
6,destination_rain_5mm,10,5608,2.210
7,item_count,4,545,0.232


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_15068\327259660.py:133: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.3. Functional Form Sensitivity

The residual screening suggests that the initial linear
terms may not adequately represent some relationships.

An alternative logistic specification is examined using
log1p transformations for distance, item value, freight,
product weight, product volume and destination rainfall.

The estimated delivery window is represented by a
centered linear term and its square. Other predictors
retain their previous coding.

The same transformations are used for shared predictors
in M0 and M4. Both specifications retain the same orders.

Convergence, residual patterns and AIC are considered
together. A lower AIC alone does not establish that a
model is adequate.

These alternatives were developed after inspecting the
training data and are treated as exploratory sensitivity
analyses.

In [12]:
log_transformations = {
    "distance_100km": "log1p_distance_100km",
    "items_value_100brl": "log1p_items_value_100brl",
    "freight_value_10brl": "log1p_freight_value_10brl",
    "product_weight_kg": "log1p_product_weight_kg",
    "product_volume_10l": "log1p_product_volume_10l",
    "destination_rain_5mm": "log1p_destination_rain_5mm",
}

for source, transformed in log_transformations.items():
    assert model_data[source].ge(0).all()
    model_data[transformed] = np.log1p(model_data[source])

# Center at 25 days and express differences in units of 10 days.
model_data["window_centered_10days"] = (
    model_data["estimated_window_days"] - 25
) / 10

model_data["window_centered_10days_sq"] = (
    model_data["window_centered_10days"] ** 2
)

m0_transformed_formula = (
    'is_late ~ log1p_distance_100km'
    ' + C(customer_state, Treatment(reference="SP"))'
    ' + item_count'
    ' + is_multi_seller'
    ' + C(purchase_weekday, Treatment(reference="Monday"))'
    ' + window_centered_10days'
    ' + window_centered_10days_sq'
)

m4_transformed_formula = (
    m0_transformed_formula
    + " + log1p_items_value_100brl"
    + " + log1p_freight_value_10brl"
    + " + log1p_product_weight_kg"
    + " + log1p_product_volume_10l"
    + " + purchase_is_national_holiday"
    + " + national_holiday_count_in_estimated_window"
    + " + log1p_destination_rain_5mm"
)

print("Alternative formulas prepared.")
print("Analysis orders:", len(model_data))

Alternative formulas prepared.
Analysis orders: 56553


In [13]:
form_sensitivity_results = {}
form_sensitivity_designs = {}
form_sensitivity_warnings = {}

alternative_formulas = {
    "M0_transformed": m0_transformed_formula,
    "M4_transformed": m4_transformed_formula,
}

for name, formula in alternative_formulas.items():
    y_check, X_check = dmatrices(
        formula,
        data=model_data,
        return_type="dataframe",
        NA_action="raise",
    )

    assert y_check.equals(y_m0)
    assert X_check.index.equals(X_m0.index)
    assert np.isfinite(X_check.to_numpy()).all()

    rank = np.linalg.matrix_rank(X_check.to_numpy())
    assert rank == X_check.shape[1], (
        f"{name}: exact linear dependencies detected."
    )

    with warnings.catch_warnings(record=True) as fit_warnings:
        warnings.simplefilter("always")

        result = sm.GLM(
            y_check.iloc[:, 0],
            X_check,
            family=sm.families.Binomial(
                link=sm.families.links.Logit()
            ),
            missing="raise",
        ).fit(
            maxiter=100,
            tol=1e-8,
        )

    form_sensitivity_results[name] = result
    form_sensitivity_designs[name] = X_check
    form_sensitivity_warnings[name] = [
        f"{warning.category.__name__}: {warning.message}"
        for warning in fit_warnings
    ]

m0_transformed = form_sensitivity_results["M0_transformed"]
m4_transformed = form_sensitivity_results["M4_transformed"]

comparison_rows = []

models_to_compare = {
    "M0_initial": m0_initial,
    "M0_transformed": m0_transformed,
    "M4_initial": m4_initial,
    "M4_transformed": m4_transformed,
}

for name, result in models_to_compare.items():
    fitted = np.asarray(result.fittedvalues)

    comparison_rows.append({
        "model": name,
        "orders": int(result.nobs),
        "parameters": len(result.params),
        "converged": bool(result.converged),
        "iterations": result.fit_history.get("iteration"),
        "finite_coefficients": bool(
            np.isfinite(result.params).all()
        ),
        "finite_standard_errors": bool(
            np.isfinite(result.bse).all()
        ),
        "aic": result.aic,
        "minimum_fitted_probability": fitted.min(),
        "maximum_fitted_probability": fitted.max(),
    })

form_comparison = pd.DataFrame(comparison_rows)

print("FUNCTIONAL FORM COMPARISON")
display(form_comparison.set_index("model").T)

print("\nALTERNATIVE MODEL WARNINGS")
for name, messages in form_sensitivity_warnings.items():
    print(f"\n{name}")
    if messages:
        for message in messages:
            print(message)
    else:
        print("No warnings were emitted.")

FUNCTIONAL FORM COMPARISON


model,M0_initial,M0_transformed,M4_initial,M4_transformed
orders,56553,56553,56553,56553
parameters,37,38,44,45
converged,True,True,True,True
iterations,7,7,7,7
finite_coefficients,True,True,True,True
finite_standard_errors,True,True,True,True
aic,25669.389657,25599.543318,25538.185694,25379.622937
minimum_fitted_probability,0.000049,0.0,0.000035,0.0
maximum_fitted_probability,0.449441,0.392912,0.676023,0.47039



ALTERNATIVE MODEL WARNINGS

M0_transformed
No warnings were emitted.

M4_transformed
No warnings were emitted.


In [14]:
boundary_rows = []

for name, result in {
    "M0_transformed": m0_transformed,
    "M4_transformed": m4_transformed,
}.items():
    probabilities = np.asarray(
        result.fittedvalues,
        dtype=float,
    )
    linear_predictor = (
        result.model.exog @ np.asarray(result.params)
    )

    assert np.isfinite(probabilities).all()
    assert (
        (probabilities >= 0) & (probabilities <= 1)
    ).all()

    boundary_rows.append({
        "model": name,
        "minimum_probability_scientific": (
            f"{probabilities.min():.6e}"
        ),
        "exact_zero_probabilities": int(
            (probabilities == 0).sum()
        ),
        "exact_one_probabilities": int(
            (probabilities == 1).sum()
        ),
        "probabilities_below_1e_minus_8": int(
            (probabilities < 1e-8).sum()
        ),
        "minimum_linear_predictor": linear_predictor.min(),
        "maximum_linear_predictor": linear_predictor.max(),
    })

probability_boundary_checks = pd.DataFrame(boundary_rows)

print("TRANSFORMED MODEL PROBABILITY CHECKS")
display(probability_boundary_checks.set_index("model").T)

TRANSFORMED MODEL PROBABILITY CHECKS


model,M0_transformed,M4_transformed
minimum_probability_scientific,5.015734e-09,2.217086e-09
exact_zero_probabilities,0,0
exact_one_probabilities,0,0
probabilities_below_1e_minus_8,1,1
minimum_linear_predictor,-19.110686,-19.927072
maximum_linear_predictor,-0.435088,-0.118578


### 5.4. Residual Comparison Across Specifications

Binned residuals from the initial and transformed M4
specifications are compared using identical groups
defined on the original predictor scales.

This preserves the same observations within each group.
Changes in the plots therefore reflect differences
between the fitted specifications.

Residual patterns and numerical stability are reviewed
alongside AIC. Neither measure alone establishes
that the model is adequate.

In [15]:
m4_transformed_fitted = np.asarray(
    m4_transformed.fittedvalues,
    dtype=float,
)

transformed_residual_tables = []

for term in screening_terms:
    check_data = pd.DataFrame({
        "value": model_data[term].to_numpy(dtype=float),
        "is_late": y_m4.iloc[:, 0].to_numpy(),
        "fitted_probability": m4_transformed_fitted,
    })

    if term == "item_count":
        check_data["band"] = pd.cut(
            check_data["value"],
            bins=[0, 1, 2, 3, np.inf],
            labels=["1", "2", "3", "4+"],
        )
    else:
        check_data["band"] = pd.qcut(
            check_data["value"],
            q=10,
            duplicates="drop",
        )

    assert check_data["band"].notna().all()

    grouped = (
        check_data.groupby("band", observed=True)
        .agg(
            orders=("is_late", "size"),
            late_orders=("is_late", "sum"),
            median_value=("value", "median"),
            observed_rate=("is_late", "mean"),
            fitted_rate=("fitted_probability", "mean"),
        )
        .reset_index()
    )

    grouped["residual_pp"] = (
        grouped["observed_rate"] - grouped["fitted_rate"]
    ) * 100

    grouped["term"] = term
    grouped["band"] = grouped["band"].astype(str)

    assert grouped["orders"].sum() == len(model_data)
    transformed_residual_tables.append(grouped)

binned_residuals_m4_transformed = pd.concat(
    transformed_residual_tables,
    ignore_index=True,
)

residual_form_comparison = binned_residuals_m4.merge(
    binned_residuals_m4_transformed,
    on=["term", "band"],
    how="outer",
    suffixes=("_initial", "_transformed"),
    validate="one_to_one",
    indicator=True,
)

assert residual_form_comparison["_merge"].eq("both").all()
assert (
    residual_form_comparison["orders_initial"]
    == residual_form_comparison["orders_transformed"]
).all()
assert (
    residual_form_comparison["late_orders_initial"]
    == residual_form_comparison["late_orders_transformed"]
).all()

residual_summary_rows = []

for term in screening_terms:
    rows = residual_form_comparison.loc[
        residual_form_comparison["term"].eq(term)
    ]

    residual_summary_rows.append({
        "term": term,
        "initial_max_absolute_residual_pp": (
            rows["residual_pp_initial"].abs().max()
        ),
        "transformed_max_absolute_residual_pp": (
            rows["residual_pp_transformed"].abs().max()
        ),
    })

residual_summary_comparison = pd.DataFrame(
    residual_summary_rows
)

print("M4 RESIDUAL COMPARISON")
display(residual_summary_comparison.round(3))

print("\nPASS: Both specifications use identical residual groups.")

M4 RESIDUAL COMPARISON


,term,initial_max_absolute_residual_pp,transformed_max_absolute_residual_pp
0,distance_100km,1.675,0.872
1,estimated_window_days,1.589,0.868
2,items_value_100brl,1.157,0.890
3,freight_value_10brl,1.717,0.974
4,product_weight_kg,0.957,0.534
5,product_volume_10l,1.015,0.572
6,destination_rain_5mm,2.210,1.832
7,item_count,0.232,0.457



PASS: Both specifications use identical residual groups.


In [16]:
fig_residual_comparison, axes = plt.subplots(
    4, 2,
    figsize=(15, 17),
    constrained_layout=True,
)

for ax, (term, title) in zip(
    axes.flat, screening_terms.items()
):
    rows = residual_form_comparison.loc[
        residual_form_comparison["term"].eq(term)
    ].sort_values("median_value_initial")

    positions = np.arange(len(rows))

    if term == "item_count":
        labels = rows["band"].tolist()
    else:
        labels = [
            f"{value:.2g}"
            for value in rows["median_value_initial"]
        ]

    ax.plot(
        positions,
        rows["residual_pp_initial"],
        marker="o",
        label="Initial M4",
    )
    ax.plot(
        positions,
        rows["residual_pp_transformed"],
        marker="s",
        label="Transformed M4",
    )

    ax.axhline(0, color="gray", linestyle="--")
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, rotation=45)
    ax.set_title(title)
    ax.set_xlabel(
        "Item-count group" if term == "item_count"
        else "Median value within quantile group"
    )
    ax.set_ylabel("Mean residual (percentage points)")
    ax.grid(axis="y", alpha=0.25)

axes.flat[0].legend()

fig_residual_comparison.suptitle(
    "M4 residuals: initial and transformed specifications",
    fontsize=15,
)

plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_15068\1092810849.py:55: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5.5. Additional Stability Checks

Multicollinearity is reassessed after transforming
the predictors.

Leverage and approximate Cook's distance are inspected
to identify observations that may strongly affect
the fitted coefficients.

Screening thresholds are used for review only.
Flagged observations are retained.

Monthly residual summaries are also examined because
orders may share temporal and weather conditions.
Persistent temporal patterns may indicate remaining
model misspecification.

In [17]:
vif_m0_transformed = numeric_term_vif(
    "M0_transformed",
    form_sensitivity_designs["M0_transformed"],
)

vif_m4_transformed = numeric_term_vif(
    "M4_transformed",
    form_sensitivity_designs["M4_transformed"],
)

print("TRANSFORMED M0 VIF")
display(
    vif_m0_transformed
    .sort_values("vif", ascending=False)
    .round(3)
)

print("\nTRANSFORMED M4 VIF")
display(
    vif_m4_transformed
    .sort_values("vif", ascending=False)
    .round(3)
)

TRANSFORMED M0 VIF


,model,term,vif,flag_vif_above_5
0,M0_transformed,log1p_distance_100km,3.081,False
3,M0_transformed,window_centered_10days,2.518,False
4,M0_transformed,window_centered_10days_sq,1.605,False
2,M0_transformed,is_multi_seller,1.065,False
1,M0_transformed,item_count,1.065,False



TRANSFORMED M4 VIF


,model,term,vif,flag_vif_above_5
6,M4_transformed,log1p_freight_value_10brl,4.094,False
7,M4_transformed,log1p_product_weight_kg,3.672,False
0,M4_transformed,log1p_distance_100km,3.507,False
8,M4_transformed,log1p_product_volume_10l,2.883,False
3,M4_transformed,window_centered_10days,2.725,False
4,M4_transformed,window_centered_10days_sq,1.614,False
1,M4_transformed,item_count,1.600,False
5,M4_transformed,log1p_items_value_100brl,1.468,False
10,M4_transformed,national_holiday_count_in_estimated_window,1.099,False
2,M4_transformed,is_multi_seller,1.068,False


In [18]:
m4_influence = m4_transformed.get_influence(
    observed=False
)

influence_checks = model_data[
    [
        "order_id",
        "customer_state",
        "is_late",
        "estimated_window_days",
        "distance_km_mean",
        "destination_rain_state_mean_mm",
    ]
].copy()

influence_checks["fitted_probability"] = (
    m4_transformed_fitted
)
influence_checks["leverage"] = (
    m4_influence.hat_matrix_diag
)
influence_checks["cooks_distance"] = (
    m4_influence.cooks_distance[0]
)

assert np.isfinite(
    influence_checks[
        ["leverage", "cooks_distance"]
    ].to_numpy()
).all()

n_analysis = len(influence_checks)
n_parameters = len(m4_transformed.params)

# Review thresholds, not deletion rules.
cook_review_threshold = 4 / n_analysis
leverage_review_threshold = (
    2 * n_parameters / n_analysis
)

influence_checks["flag_cook_review"] = (
    influence_checks["cooks_distance"]
    > cook_review_threshold
)
influence_checks["flag_leverage_review"] = (
    influence_checks["leverage"]
    > leverage_review_threshold
)

influence_summary = pd.DataFrame([{
    "orders": n_analysis,
    "parameters": n_parameters,
    "cook_review_threshold": cook_review_threshold,
    "leverage_review_threshold": leverage_review_threshold,
    "orders_flagged_cook": int(
        influence_checks["flag_cook_review"].sum()
    ),
    "orders_flagged_leverage": int(
        influence_checks["flag_leverage_review"].sum()
    ),
    "maximum_cooks_distance": (
        influence_checks["cooks_distance"].max()
    ),
    "maximum_leverage": (
        influence_checks["leverage"].max()
    ),
}])

print("M4 INFLUENCE SUMMARY")
display(influence_summary.T)

print("\nTEN ORDERS WITH THE LARGEST COOK'S DISTANCE")
display(
    influence_checks
    .nlargest(10, "cooks_distance")
    .round(6)
)

print("\nAll observations are retained.")

M4 INFLUENCE SUMMARY


,0
orders,56553.000000
parameters,45.000000
cook_review_threshold,0.000071
leverage_review_threshold,0.001591
orders_flagged_cook,3327.000000
orders_flagged_leverage,5513.000000
maximum_cooks_distance,0.025608
maximum_leverage,0.217781



TEN ORDERS WITH THE LARGEST COOK'S DISTANCE


,order_id,customer_state,is_late,estimated_window_days,distance_km_mean,destination_rain_state_mean_mm,fitted_probability,leverage,cooks_distance,flag_cook_review,flag_leverage_review
5181,2fa29503f2ebd9f53deba187160f3202,RR,1,40,3301.504600,28.400000,0.180575,0.173479,0.025608,True,True
7824,28f12a3e5117f5e70bbdb45be9188639,AM,1,35,2367.138420,11.187500,0.027058,0.026975,0.022766,True,True
49877,5d1e025052895991d3f50688be8f626e,RO,1,39,2468.894308,0.200000,0.006609,0.006667,0.022570,True,True
16047,8272b63d03f5f79c56e9e4120aec44ef,SP,1,12,38.532463,0.035294,0.005534,0.003546,0.014261,True,True
22520,2bcf1f79964f8b4f4f62f19441601b1a,AC,1,43,3228.179821,0.200000,0.085469,0.043361,0.011266,True,True
4179,46e986bd2d3cdddfbc642749d67b4777,AP,1,43,2646.448949,25.900000,0.155928,0.075614,0.010645,True,True
3915,437222e3fd1b07396f1d9ba8c15fba59,AP,1,43,2435.055623,21.000000,0.165869,0.079212,0.010441,True,True
33629,5f082c0821ef7e1812362878c7268aef,AC,1,40,2716.467696,8.200000,0.171063,0.078552,0.009962,True,True
45165,f939e2a89376e44158657a15465cacce,RJ,1,70,326.296218,30.872727,0.005794,0.001722,0.006588,True,True
42413,1c1975ccdbd34c48a241f30cc2258dbb,PE,1,64,2108.785002,0.000000,0.005985,0.001012,0.003744,True,False



All observations are retained.


In [19]:
purchase_times_check = pd.to_datetime(
    model_data["order_purchase_timestamp"],
    format="ISO8601",
    errors="raise",
    utc=True,
)

temporal_check_data = pd.DataFrame({
    "purchase_month": (
        purchase_times_check.dt.strftime("%Y-%m")
        .to_numpy()
    ),
    "is_late": model_data["is_late"].to_numpy(),
    "fitted_probability": m4_transformed_fitted,
    "rain_index_mm": (
        model_data["destination_rain_state_mean_mm"]
        .to_numpy()
    ),
})

monthly_residual_checks = (
    temporal_check_data.groupby("purchase_month")
    .agg(
        orders=("is_late", "size"),
        late_orders=("is_late", "sum"),
        observed_rate=("is_late", "mean"),
        fitted_rate=("fitted_probability", "mean"),
        mean_rain_index_mm=("rain_index_mm", "mean"),
    )
    .reset_index()
)

monthly_residual_checks["observed_percent"] = (
    monthly_residual_checks["observed_rate"] * 100
)
monthly_residual_checks["fitted_percent"] = (
    monthly_residual_checks["fitted_rate"] * 100
)
monthly_residual_checks["residual_pp"] = (
    monthly_residual_checks["observed_percent"]
    - monthly_residual_checks["fitted_percent"]
)

assert monthly_residual_checks["orders"].sum() == n_analysis

print("M4 RESIDUALS BY PURCHASE MONTH")
display(
    monthly_residual_checks[
        [
            "purchase_month",
            "orders",
            "late_orders",
            "observed_percent",
            "fitted_percent",
            "residual_pp",
            "mean_rain_index_mm",
        ]
    ].round(3)
)

M4 RESIDUALS BY PURCHASE MONTH


,purchase_month,orders,late_orders,observed_percent,fitted_percent,residual_pp,mean_rain_index_mm
0,2016-09,1,1,100.000,4.580,95.420,0.387
1,2016-10,261,2,0.766,0.528,0.239,3.978
2,2016-12,1,0,0.000,1.437,-1.437,2.723
3,2017-01,739,21,2.842,2.663,0.179,7.167
4,2017-02,1618,49,3.028,4.473,-1.445,4.530
5,2017-03,2500,113,4.520,7.014,-2.494,4.791
6,2017-04,2247,143,6.364,5.612,0.752,3.836
7,2017-05,3497,104,2.974,6.608,-3.634,4.175
8,2017-06,3078,93,3.021,6.158,-3.136,2.616
9,2017-07,3829,100,2.612,6.245,-3.634,0.797


In [20]:
# Anchor both block definitions to the same Monday.
first_purchase = purchase_times_check.min().normalize()
block_anchor = first_purchase - pd.Timedelta(
    days=first_purchase.weekday()
)

time_block_ids = {}
time_block_tables = {}
block_summary_rows = []

for width in [7, 14]:
    block_ids = (
        (purchase_times_check.dt.normalize() - block_anchor)
        .dt.days // width
    ).astype(int)

    time_block_ids[width] = block_ids.to_numpy()

    block_data = pd.DataFrame({
        "block_id": block_ids.to_numpy(),
        "purchase_time": purchase_times_check.to_numpy(),
        "is_late": model_data["is_late"].to_numpy(),
    })

    block_table = (
        block_data.groupby("block_id")
        .agg(
            orders=("is_late", "size"),
            late_orders=("is_late", "sum"),
            first_purchase=("purchase_time", "min"),
            last_purchase=("purchase_time", "max"),
        )
        .reset_index()
    )

    assert block_table["orders"].sum() == len(model_data)
    assert block_table["late_orders"].sum() == 3649

    time_block_tables[width] = block_table

    block_summary_rows.append({
        "block_days": width,
        "nonempty_blocks": len(block_table),
        "smallest_block_orders": block_table["orders"].min(),
        "median_block_orders": block_table["orders"].median(),
        "largest_block_orders": block_table["orders"].max(),
        "blocks_without_late_orders": int(
            block_table["late_orders"].eq(0).sum()
        ),
    })

time_block_summary = pd.DataFrame(block_summary_rows)

print("TIME BLOCK ANCHOR")
print(block_anchor)

print("\nTEMPORAL BLOCK SUMMARY")
display(time_block_summary)

print("\nNo predictors or model specifications were changed.")

TIME BLOCK ANCHOR
2016-09-12 00:00:00+00:00

TEMPORAL BLOCK SUMMARY


,block_days,nonempty_blocks,smallest_block_orders,median_block_orders,largest_block_orders,blocks_without_late_orders
0,7,65,1,862.0,2883,2
1,14,35,1,1644.0,4896,2



No predictors or model specifications were changed.


### 5.6. Sensitivity Analysis with Calendar Adjustment

The initial models showed substantial residual
differences across purchase months. Some periods had
higher observed late-delivery rates than the models
estimated.

Weather also varies over time. Its estimated association
with late delivery may therefore partly reflect
differences between purchase periods.

A supplementary analysis adds purchase-period
indicators to both transformed M0 and M4.
This adjustment allows the baseline odds of late
delivery to differ between periods.

Purchases in 2016 are pooled into one category because
individual months contain very few observations.
From January 2017 onward, each year-month is retained.
January 2017 is the reference period.

The shared protocol reserves year-month for splitting
and auditing. Its use here is an explicit extension
for Stage 4 sensitivity analysis. The shared predictor
list, source dataset and split manifest are unchanged.

All models use the same 56,553 orders.
The earlier specifications are retained for comparison.

This analysis was added after inspecting training
residuals. It is exploratory and does not establish
causation or eliminate all possible confounding.

Closer agreement within training periods is expected
when period indicators are included. It is not an
independent validation of predictive performance.

In [21]:
assert purchase_times_check.index.equals(model_data.index)

model_data["purchase_period"] = (
    purchase_times_check.dt.strftime("%Y-%m")
    .astype(object)
)

model_data.loc[
    purchase_times_check.dt.year.eq(2016),
    "purchase_period",
] = "2016 (pooled)"

period_sample_check = outcome_counts(
    model_data,
    "purchase_period",
).sort_values("purchase_period")

assert model_data["purchase_period"].notna().all()
assert model_data["purchase_period"].nunique() == 16
assert "2017-01" in model_data["purchase_period"].unique()
assert period_sample_check["orders"].sum() == 56553
assert period_sample_check["late_orders"].sum() == 3649

print("PURCHASE PERIOD SAMPLE CHECK")
display(period_sample_check.round(3))

print(
    "\nPeriods containing only one outcome:",
    int(period_sample_check["single_outcome_group"].sum()),
)

PURCHASE PERIOD SAMPLE CHECK


,purchase_period,orders,late_orders,not_late_orders,late_percent,single_outcome_group
0,2016 (pooled),263,3,260,1.141,False
1,2017-01,739,21,718,2.842,False
2,2017-02,1618,49,1569,3.028,False
3,2017-03,2500,113,2387,4.520,False
4,2017-04,2247,143,2104,6.364,False
5,2017-05,3497,104,3393,2.974,False
6,2017-06,3078,93,2985,3.021,False
7,2017-07,3829,100,3729,2.612,False
8,2017-08,4150,120,4030,2.892,False
9,2017-09,4059,175,3884,4.311,False



Periods containing only one outcome: 0


In [22]:
period_term = (
    'C(purchase_period, Treatment(reference="2017-01"))'
)

time_adjusted_formulas = {
    "M0_time_adjusted": (
        m0_transformed_formula + " + " + period_term
    ),
    "M4_time_adjusted": (
        m4_transformed_formula + " + " + period_term
    ),
}

time_adjusted_results = {}
time_adjusted_designs = {}
time_adjusted_warning_messages = {}
time_adjusted_check_rows = []

for name, formula in time_adjusted_formulas.items():
    y_time, X_time = dmatrices(
        formula,
        data=model_data,
        return_type="dataframe",
        NA_action="raise",
    )

    assert y_time.equals(y_m0)
    assert X_time.index.equals(X_m0.index)
    assert np.isfinite(X_time.to_numpy()).all()

    design_rank = np.linalg.matrix_rank(
        X_time.to_numpy()
    )

    assert design_rank == X_time.shape[1], (
        f"{name}: exact linear dependencies detected."
    )

    with warnings.catch_warnings(record=True) as fit_warnings:
        warnings.simplefilter("always")

        result = sm.GLM(
            y_time.iloc[:, 0],
            X_time,
            family=sm.families.Binomial(
                link=sm.families.links.Logit()
            ),
            missing="raise",
        ).fit(
            maxiter=100,
            tol=1e-8,
        )

    time_adjusted_results[name] = result
    time_adjusted_designs[name] = X_time
    time_adjusted_warning_messages[name] = [
        f"{warning.category.__name__}: {warning.message}"
        for warning in fit_warnings
    ]

    time_adjusted_check_rows.append({
        "model": name,
        "orders": int(result.nobs),
        "parameters": len(result.params),
        "design_rank": design_rank,
        "converged": bool(result.converged),
        "iterations": result.fit_history.get("iteration"),
        "finite_coefficients": bool(
            np.isfinite(result.params).all()
        ),
        "finite_standard_errors": bool(
            np.isfinite(result.bse).all()
        ),
        "aic": result.aic,
        "warning_count": len(fit_warnings),
    })

m0_time_adjusted = time_adjusted_results["M0_time_adjusted"]
m4_time_adjusted = time_adjusted_results["M4_time_adjusted"]

time_adjusted_checks = pd.DataFrame(
    time_adjusted_check_rows
)

print("TIME-ADJUSTED MODEL CHECKS")
display(time_adjusted_checks.set_index("model").T)

print("\nFIT WARNINGS")
for name, messages in time_adjusted_warning_messages.items():
    print(f"\n{name}")
    if messages:
        for message in messages:
            print(message)
    else:
        print("No warnings were emitted.")

TIME-ADJUSTED MODEL CHECKS


model,M0_time_adjusted,M4_time_adjusted
orders,56553,56553
parameters,53,60
design_rank,53,60
converged,True,True
iterations,7,7
finite_coefficients,True,True
finite_standard_errors,True,True
aic,24290.885681,24160.994139
warning_count,0,0



FIT WARNINGS

M0_time_adjusted
No warnings were emitted.

M4_time_adjusted
No warnings were emitted.


In [23]:
comparison_terms = [
    "log1p_distance_100km",
    "log1p_freight_value_10brl",
    "purchase_is_national_holiday",
    "national_holiday_count_in_estimated_window",
    "log1p_destination_rain_5mm",
]

calendar_coefficient_comparison = pd.DataFrame({
    "without_period_adjustment": (
        m4_transformed.params.loc[comparison_terms]
    ),
    "with_period_adjustment": (
        m4_time_adjusted.params.loc[comparison_terms]
    ),
})

calendar_coefficient_comparison.index.name = "term"

print("M4 COEFFICIENTS BEFORE AND AFTER CALENDAR ADJUSTMENT")
display(calendar_coefficient_comparison.round(6))

M4 COEFFICIENTS BEFORE AND AFTER CALENDAR ADJUSTMENT


,without_period_adjustment,with_period_adjustment
term,,
log1p_distance_100km,0.434294,0.471362
log1p_freight_value_10brl,0.170265,0.081412
purchase_is_national_holiday,-0.522909,-0.670204
national_holiday_count_in_estimated_window,-0.026695,-0.103917
log1p_destination_rain_5mm,0.366826,0.051105


### 5.7. Findings from Calendar Adjustment

Both calendar-adjusted models converged on the same
56,553 orders without fitting warnings.

In M4, the coefficient of log1p destination rainfall
decreased from 0.366826 to 0.051105 after adjustment
for purchase period.

The freight and holiday coefficients also changed.
The estimated associations are therefore sensitive
to calendar adjustment.

This comparison does not establish that rainfall has
no association with late delivery. Uncertainty must
be evaluated before interpreting the estimates.

Changes in logistic coefficients do not directly
measure the proportion of an association explained
by time. The analysis remains observational and
exploratory.

## 6. Clustered Uncertainty Estimation

Cluster-robust covariance estimates account for
possible dependence among orders within the same
time block.

The main uncertainty estimates use 65 nonempty
seven-day blocks. A sensitivity analysis uses
35 nonempty fourteen-day blocks.

Both definitions share the same UTC anchor:
12 September 2016.

A finite-sample covariance correction is applied.
Confidence intervals and p-values use a t distribution
with the number of clusters minus one degrees
of freedom.

This approach allows dependence within blocks but
assumes independence between blocks. Dependence
across blocks may remain.

The fourteen-day analysis has relatively few clusters.
Its covariance matrix may be rank deficient, so it
is not used for a joint test of all coefficients.

Cluster adjustment does not correct omitted variables,
functional-form errors or sparse outcome groups.
Intervals are conditional on the fitted specifications
and do not account for exploratory model selection.

In [24]:
inference_models = {
    "M0_transformed": (
        m0_transformed,
        form_sensitivity_designs["M0_transformed"],
    ),
    "M4_transformed": (
        m4_transformed,
        form_sensitivity_designs["M4_transformed"],
    ),
    "M0_time_adjusted": (
        m0_time_adjusted,
        time_adjusted_designs["M0_time_adjusted"],
    ),
    "M4_time_adjusted": (
        m4_time_adjusted,
        time_adjusted_designs["M4_time_adjusted"],
    ),
}

clustered_results = {}
cluster_fit_check_rows = []
cluster_warning_messages = {}

for block_days in [7, 14]:
    groups = time_block_ids[block_days]
    n_clusters = len(np.unique(groups))

    assert len(groups) == len(model_data)

    for name, (reference_result, design) in inference_models.items():
        assert design.index.equals(model_data.index)

        with warnings.catch_warnings(record=True) as fit_warnings:
            warnings.simplefilter("always")

            result = sm.GLM(
                y_m4.iloc[:, 0],
                design,
                family=sm.families.Binomial(
                    link=sm.families.links.Logit()
                ),
                missing="raise",
            ).fit(
                start_params=reference_result.params.to_numpy(),
                maxiter=100,
                tol=1e-8,
                cov_type="cluster",
                cov_kwds={
                    "groups": groups,
                    "use_correction": True,
                    "df_correction": True,
                },
                use_t=True,
            )

        key = (name, block_days)
        clustered_results[key] = result

        cluster_warning_messages[key] = [
            f"{warning.category.__name__}: {warning.message}"
            for warning in fit_warnings
        ]

        coefficient_difference = float(
            np.max(
                np.abs(
                    result.params.to_numpy()
                    - reference_result.params.to_numpy()
                )
            )
        )

        covariance = np.asarray(result.cov_params())
        finite_covariance = bool(
            np.isfinite(covariance).all()
        )
        inference_df = getattr(
            result,
            "df_resid_inference",
            np.nan,
        )

        np.testing.assert_allclose(
            result.params.to_numpy(),
            reference_result.params.to_numpy(),
            rtol=1e-6,
            atol=1e-6,
        )

        assert int(result.nobs) == 56553
        assert inference_df == n_clusters - 1
        assert result.use_t

        cluster_fit_check_rows.append({
            "model": name,
            "block_days": block_days,
            "clusters": n_clusters,
            "inference_df": inference_df,
            "parameters": len(result.params),
            "converged": bool(result.converged),
            "max_coefficient_difference": coefficient_difference,
            "finite_covariance": finite_covariance,
            "finite_standard_errors": bool(
                np.isfinite(result.bse).all()
            ),
            "covariance_rank": (
                np.linalg.matrix_rank(covariance)
                if finite_covariance else np.nan
            ),
            "warning_count": len(fit_warnings),
        })

cluster_fit_checks = pd.DataFrame(
    cluster_fit_check_rows
)

print("CLUSTERED UNCERTAINTY CHECKS")
display(cluster_fit_checks)

print("\nFIT WARNINGS")
any_warnings = False

for (name, block_days), messages in cluster_warning_messages.items():
    if messages:
        any_warnings = True
        print(f"\n{name}, {block_days}-day blocks")
        for message in messages:
            print(message)

if not any_warnings:
    print("No warnings were emitted.")

CLUSTERED UNCERTAINTY CHECKS


,model,block_days,clusters,inference_df,parameters,converged,max_coefficient_difference,finite_covariance,finite_standard_errors,covariance_rank,warning_count
0,M0_transformed,7,65,64,38,True,8.333143e-10,True,True,38,0
1,M4_transformed,7,65,64,45,True,9.541921e-10,True,True,45,0
2,M0_time_adjusted,7,65,64,53,True,1.563408e-09,True,True,52,0
3,M4_time_adjusted,7,65,64,60,True,1.867953e-09,True,True,59,0
4,M0_transformed,14,35,34,38,True,8.333143e-10,True,True,34,0
5,M4_transformed,14,35,34,45,True,9.541921e-10,True,True,34,0
6,M0_time_adjusted,14,35,34,53,True,1.563408e-09,True,True,36,0
7,M4_time_adjusted,14,35,34,60,True,1.867953e-09,True,True,35,0



FIT WARNINGS
No warnings were emitted.


## 7. Adjusted Odds Ratios

Adjusted odds ratios are reported for explicitly
defined predictor contrasts while holding the other
model terms fixed.

For log-transformed predictors, contrasts are calculated
using differences on the transformed scale.

The delivery-window contrast includes both its linear
and quadratic terms.

Confidence intervals use the clustered covariance
estimates and the corresponding t distribution.
Reported p-values test whether the contrast OR equals 1.

Intervals and p-values are pointwise and are not
adjusted for multiple comparisons. They are interpreted
as exploratory results alongside effect sizes,
diagnostics and sensitivity analyses.

Odds ratios are not risk ratios and do not establish
causal effects.

In [25]:
def log1p_change(reference, comparison, scale=1):
    return (
        np.log1p(comparison / scale)
        - np.log1p(reference / scale)
    )


contrast_specs = [
    (
        "distance_km_mean",
        "200 vs 100 km",
        {"log1p_distance_100km": log1p_change(100, 200, 100)},
    ),
    (
        "item_count",
        "2 vs 1 items",
        {"item_count": 1},
    ),
    (
        "seller_count",
        "Multiple sellers vs one seller",
        {"is_multi_seller": 1},
    ),
    (
        "estimated_window_days",
        "30 vs 25 days",
        {
            "window_centered_10days": 0.5,
            "window_centered_10days_sq": 0.25,
        },
    ),
    (
        "items_value",
        "150 vs 50 BRL",
        {"log1p_items_value_100brl": log1p_change(50, 150, 100)},
    ),
    (
        "freight_value",
        "20 vs 10 BRL",
        {"log1p_freight_value_10brl": log1p_change(10, 20, 10)},
    ),
    (
        "total_product_weight_g",
        "1.5 vs 0.5 kg",
        {"log1p_product_weight_kg": log1p_change(0.5, 1.5)},
    ),
    (
        "total_product_volume_cm3",
        "15000 vs 5000 cm3",
        {"log1p_product_volume_10l": log1p_change(5000, 15000, 10000)},
    ),
    (
        "purchase_is_national_holiday",
        "Holiday vs non-holiday",
        {"purchase_is_national_holiday": 1},
    ),
    (
        "national_holiday_count_in_estimated_window",
        "1 vs 0 holidays in estimated window",
        {"national_holiday_count_in_estimated_window": 1},
    ),
    (
        "destination_rain_state_mean_mm",
        "5 vs 0 mm rainfall index",
        {"log1p_destination_rain_5mm": log1p_change(0, 5, 5)},
    ),
]

In [26]:
def calculate_contrast_or(result, weights):
    contrast = np.zeros(len(result.params), dtype=float)

    for term, weight in weights.items():
        position = result.params.index.get_loc(term)
        contrast[position] = weight

    covariance = np.asarray(result.cov_params())
    variance = float(contrast @ covariance @ contrast)

    assert np.isfinite(variance) and variance > 0, (
        "The contrast variance is not finite and positive."
    )

    test = result.t_test(contrast, use_t=True)

    log_or = np.asarray(test.effect).item()
    log_ci_lower, log_ci_upper = np.asarray(
        test.conf_int(alpha=0.05)
    ).reshape(-1, 2)[0]

    assert test.df_denom == result.df_resid_inference

    values = {
        "log_odds_ratio": log_or,
        "standard_error_log_or": np.sqrt(variance),
        "adjusted_or": np.exp(log_or),
        "ci_lower": np.exp(log_ci_lower),
        "ci_upper": np.exp(log_ci_upper),
        "p_value": np.asarray(test.pvalue).item(),
    }

    assert np.isfinite(list(values.values())).all()
    assert (
        0 < values["ci_lower"]
        <= values["adjusted_or"]
        <= values["ci_upper"]
    )
    assert 0 <= values["p_value"] <= 1

    return values


odds_ratio_rows = []

for (model_name, block_days), result in clustered_results.items():
    for feature, comparison, weights in contrast_specs:
        # A feature absent from M0 is not assigned an OR.
        if not all(term in result.params.index for term in weights):
            continue

        estimates = calculate_contrast_or(result, weights)

        odds_ratio_rows.append({
            "model": model_name,
            "block_days": block_days,
            "clusters": len(np.unique(time_block_ids[block_days])),
            "inference_df": result.df_resid_inference,
            "orders": int(result.nobs),
            "late_orders": int(result.model.endog.sum()),
            "feature": feature,
            "comparison": comparison,
            **estimates,
        })

adjusted_or_contrasts = pd.DataFrame(odds_ratio_rows)

assert not adjusted_or_contrasts.duplicated(
    ["model", "block_days", "feature", "comparison"]
).any()

assert adjusted_or_contrasts["orders"].eq(56553).all()
assert adjusted_or_contrasts["late_orders"].eq(3649).all()

print("PASS: Predictor contrast ORs and confidence intervals calculated.")

PASS: Predictor contrast ORs and confidence intervals calculated.


In [27]:
focus_features = [
    "distance_km_mean",
    "freight_value",
    "purchase_is_national_holiday",
    "national_holiday_count_in_estimated_window",
    "destination_rain_state_mean_mm",
]

for block_days in [7, 14]:
    preview = adjusted_or_contrasts.loc[
        adjusted_or_contrasts["model"].isin(
            ["M4_transformed", "M4_time_adjusted"]
        )
        & adjusted_or_contrasts["block_days"].eq(block_days)
        & adjusted_or_contrasts["feature"].isin(focus_features),
        [
            "model",
            "feature",
            "comparison",
            "adjusted_or",
            "ci_lower",
            "ci_upper",
            "p_value",
        ],
    ].copy()

    preview[["adjusted_or", "ci_lower", "ci_upper"]] = (
        preview[["adjusted_or", "ci_lower", "ci_upper"]]
        .round(4)
    )
    preview["p_value"] = preview["p_value"].map(
        lambda value: f"{value:.3g}"
    )

    print(f"\nM4 KEY CONTRASTS — {block_days}-DAY BLOCKS")
    display(preview)


M4 KEY CONTRASTS — 7-DAY BLOCKS


,model,feature,comparison,adjusted_or,ci_lower,ci_upper,p_value
4,M4_transformed,distance_km_mean,200 vs 100 km,1.1925,1.1008,1.2919,4.25e-05
9,M4_transformed,freight_value,20 vs 10 BRL,1.0715,0.9701,1.1834,0.17
12,M4_transformed,purchase_is_national_holiday,Holiday vs non-holiday,0.5928,0.4077,0.8619,0.00692
13,M4_transformed,national_holiday_count_in_estimated_window,1 vs 0 holidays in estimated window,0.9737,0.7645,1.2400,0.826
14,M4_transformed,destination_rain_state_mean_mm,5 vs 0 mm rainfall index,1.2895,1.1685,1.4231,2.65e-06
19,M4_time_adjusted,distance_km_mean,200 vs 100 km,1.2106,1.1490,1.2756,5.43e-10
24,M4_time_adjusted,freight_value,20 vs 10 BRL,1.0336,0.9263,1.1532,0.549
27,M4_time_adjusted,purchase_is_national_holiday,Holiday vs non-holiday,0.5116,0.3352,0.7810,0.00237
28,M4_time_adjusted,national_holiday_count_in_estimated_window,1 vs 0 holidays in estimated window,0.9013,0.6704,1.2118,0.486
29,M4_time_adjusted,destination_rain_state_mean_mm,5 vs 0 mm rainfall index,1.0361,0.9757,1.1001,0.242



M4 KEY CONTRASTS — 14-DAY BLOCKS


,model,feature,comparison,adjusted_or,ci_lower,ci_upper,p_value
34,M4_transformed,distance_km_mean,200 vs 100 km,1.1925,1.1038,1.2884,5.19e-05
39,M4_transformed,freight_value,20 vs 10 BRL,1.0715,0.9852,1.1653,0.104
42,M4_transformed,purchase_is_national_holiday,Holiday vs non-holiday,0.5928,0.3910,0.8986,0.0153
43,M4_transformed,national_holiday_count_in_estimated_window,1 vs 0 holidays in estimated window,0.9737,0.7415,1.2785,0.843
44,M4_transformed,destination_rain_state_mean_mm,5 vs 0 mm rainfall index,1.2895,1.1496,1.4465,7.62e-05
49,M4_time_adjusted,distance_km_mean,200 vs 100 km,1.2106,1.1452,1.2798,4.58e-08
54,M4_time_adjusted,freight_value,20 vs 10 BRL,1.0336,0.9446,1.1309,0.461
57,M4_time_adjusted,purchase_is_national_holiday,Holiday vs non-holiday,0.5116,0.3255,0.8041,0.00487
58,M4_time_adjusted,national_holiday_count_in_estimated_window,1 vs 0 holidays in estimated window,0.9013,0.6950,1.1688,0.422
59,M4_time_adjusted,destination_rain_state_mean_mm,5 vs 0 mm rainfall index,1.0361,0.9821,1.0930,0.187


### 7.1. Main Findings and Sensitivity

Distance remained positively associated with late
delivery across the examined specifications.

In calendar-adjusted M4, the adjusted OR for 200
versus 100 km was 1.2106, with a seven-day clustered
95% CI of [1.1490, 1.2756].

The rainfall association was sensitive to calendar
adjustment. For a rainfall index of 5 versus 0 mm,
the OR changed from 1.2895 [1.1685, 1.4231] to
1.0361 [0.9757, 1.1001].

The calendar-adjusted rainfall interval included 1.
This does not establish the absence of an association
or demonstrate a lack of predictive value.

Purchase on a national holiday was associated with
lower odds of late delivery. The calendar-adjusted
OR was 0.5116 [0.3352, 0.7810].
This association does not establish a protective effect.

The freight and delivery-window holiday-count
contrasts had intervals that included 1.

The main interpretations were unchanged under
fourteen-day clustering. These results remain
conditional on the sample, model specification
and clustering assumptions.

In [28]:
categorical_specs = [
    ("customer_state", "SP", "C(customer_state,"),
    ("purchase_weekday", "Monday", "C(purchase_weekday,"),
]

categorical_or_rows = []

for (model_name, block_days), result in clustered_results.items():
    for feature, reference, prefix in categorical_specs:
        category_terms = [
            term for term in result.params.index
            if term.startswith(prefix)
        ]

        for term in category_terms:
            level = term.split("[T.", 1)[1].removesuffix("]")

            level_mask = model_data[feature].eq(level)
            group_orders = int(level_mask.sum())
            group_late_orders = int(
                model_data.loc[level_mask, "is_late"].sum()
            )
            group_not_late_orders = (
                group_orders - group_late_orders
            )

            assert group_orders > 0

            sparse_group = (
                group_late_orders < 5
                or group_not_late_orders < 5
            )

            estimates = calculate_contrast_or(
                result,
                {term: 1},
            )

            categorical_or_rows.append({
                "model": model_name,
                "block_days": block_days,
                "clusters": len(
                    np.unique(time_block_ids[block_days])
                ),
                "inference_df": result.df_resid_inference,
                "orders": int(result.nobs),
                "late_orders": int(result.model.endog.sum()),
                "feature": feature,
                "comparison": f"{level} vs {reference}",
                "group_orders": group_orders,
                "group_late_orders": group_late_orders,
                "interpretation_note": (
                    "Sparse outcome group; interpret cautiously"
                    if sparse_group else ""
                ),
                **estimates,
            })

adjusted_odds_ratios = pd.concat(
    [
        adjusted_or_contrasts.assign(
            interpretation_note=""
        ),
        pd.DataFrame(categorical_or_rows),
    ],
    ignore_index=True,
)

key_columns = [
    "model",
    "block_days",
    "feature",
    "comparison",
]

assert not adjusted_odds_ratios.duplicated(
    key_columns
).any()

assert len(adjusted_odds_ratios) == 316
assert set(adjusted_odds_ratios["feature"]) == set(main_features)
assert adjusted_odds_ratios["orders"].eq(56553).all()
assert adjusted_odds_ratios["late_orders"].eq(3649).all()

print("FULL OR TABLE COUNTS")
display(
    adjusted_odds_ratios.groupby(
        ["model", "block_days"]
    ).size().rename("contrasts").reset_index()
)

print("\nSPARSE STATE GROUPS: CALENDAR-ADJUSTED M4, 7-DAY BLOCKS")

sparse_state_preview = adjusted_odds_ratios.loc[
    adjusted_odds_ratios["model"].eq("M4_time_adjusted")
    & adjusted_odds_ratios["block_days"].eq(7)
    & adjusted_odds_ratios["feature"].eq("customer_state")
    & adjusted_odds_ratios["interpretation_note"].ne(""),
    [
        "comparison",
        "group_orders",
        "group_late_orders",
        "adjusted_or",
        "ci_lower",
        "ci_upper",
    ],
]

display(sparse_state_preview.round(4))

FULL OR TABLE COUNTS


,model,block_days,contrasts
0,M0_time_adjusted,7,36
1,M0_time_adjusted,14,36
2,M0_transformed,7,36
3,M0_transformed,14,36
4,M4_time_adjusted,7,43
5,M4_time_adjusted,14,43
6,M4_transformed,7,43
7,M4_transformed,14,43



SPARSE STATE GROUPS: CALENDAR-ADJUSTED M4, 7-DAY BLOCKS


,comparison,group_orders,group_late_orders,adjusted_or,ci_lower,ci_upper
156,AC vs SP,28.0,2.0,3.1560,0.6861,14.5166
158,AM vs SP,68.0,1.0,0.7022,0.0894,5.5176
159,AP vs SP,27.0,2.0,6.8505,1.4687,31.9534
176,RO vs SP,107.0,1.0,0.3872,0.0497,3.0167
177,RR vs SP,10.0,1.0,3.4091,0.2726,42.6371


In [29]:
regression_tables_dir = (
    PROJECT_ROOT / 'outputs/tables/explanatory_regression'
)
regression_tables_dir.mkdir(
    parents=True,
    exist_ok=True,
)

adjusted_or_path = (
    regression_tables_dir / "adjusted_odds_ratios.csv"
)

adjusted_odds_ratios.to_csv(
    adjusted_or_path,
    index=False,
    encoding="utf-8-sig",
)

assert adjusted_or_path.is_file()
assert adjusted_or_path.stat().st_size > 0

print("Saved:", adjusted_or_path)
print("Rows:", len(adjusted_odds_ratios))
print("PASS: Full adjusted OR table prepared and saved.")

Saved: C:\Users\LENOVO\OneDrive\Desktop\PTTK\outputs\tables\regression\adjusted_odds_ratios.csv
Rows: 316
PASS: Full adjusted OR table prepared and saved.


## 8. Export and Reproducibility

The exported results include the analysis sample,
adjusted odds ratios, diagnostic checks and figures.

The OR table contains both model specifications and
both temporal clustering definitions.

Sparse state groups are flagged for cautious
interpretation. Calendar indicators are adjustment
terms and are excluded from the main predictor
contrast table.

In [30]:
sample_summary = sample_audit.copy()
sample_summary["dataset_sha256"] = dataset_sha256

sample_path = (
    regression_tables_dir / "sample_summary.csv"
)

sample_summary.to_csv(
    sample_path,
    index=False,
    encoding="utf-8-sig",
)


diagnostic_rows = []

metadata_columns = {
    "model",
    "block_days",
    "term",
    "feature",
    "purchase_month",
    "purchase_period",
}

detail_columns = [
    "term",
    "feature",
    "purchase_month",
    "purchase_period",
]


def record_diagnostic_table(section, table, default_model):
    for row in table.to_dict("records"):
        detail = "; ".join(
            f"{column}={row[column]}"
            for column in detail_columns
            if column in row and pd.notna(row[column])
        )

        for check, value in row.items():
            if check in metadata_columns:
                continue

            diagnostic_rows.append({
                "section": section,
                "model": row.get("model", default_model),
                "block_days": row.get("block_days"),
                "detail": detail,
                "check": check,
                "value": str(value),
            })


all_checked_vif = pd.concat(
    [
        vif_results,
        vif_m0_transformed,
        vif_m4_transformed,
    ],
    ignore_index=True,
)

diagnostic_sources = [
    (
        "predictor_missingness",
        missing_summary,
        "Shared training sample",
    ),
    (
        "initial_and_transformed_fits",
        form_comparison,
        "Shared",
    ),
    (
        "calendar_adjusted_fits",
        time_adjusted_checks,
        "Shared",
    ),
    (
        "clustered_covariance",
        cluster_fit_checks,
        "Shared",
    ),
    (
        "vif",
        all_checked_vif,
        "Shared",
    ),
    (
        "probability_boundaries",
        probability_boundary_checks,
        "Shared",
    ),
    (
        "functional_form_residuals",
        residual_summary_comparison,
        "M4_initial_vs_transformed",
    ),
    (
        "influence_screening",
        influence_summary,
        "M4_transformed",
    ),
    (
        "monthly_residuals",
        monthly_residual_checks,
        "M4_transformed",
    ),
    (
        "temporal_blocks",
        time_block_summary,
        "Shared",
    ),
    (
        "calendar_group_counts",
        period_sample_check,
        "Shared",
    ),
]

for section, table, default_model in diagnostic_sources:
    record_diagnostic_table(
        section,
        table,
        default_model,
    )

diagnostic_rows.append({
    "section": "temporal_blocks",
    "model": "Shared",
    "block_days": None,
    "detail": "",
    "check": "utc_block_anchor",
    "value": str(block_anchor),
})

diagnostics = pd.DataFrame(diagnostic_rows)

assert not diagnostics.duplicated(
    ["section", "model", "block_days", "detail", "check"]
).any()

diagnostics_path = (
    regression_tables_dir / "diagnostics.csv"
)

diagnostics.to_csv(
    diagnostics_path,
    index=False,
    encoding="utf-8-sig",
)

print("Sample summary rows:", len(sample_summary))
print("Diagnostic rows:", len(diagnostics))

Sample summary rows: 3
Diagnostic rows: 513


In [31]:
regression_figures_dir = (
    PROJECT_ROOT / "outputs" / "figures" / "regression"
)
regression_figures_dir.mkdir(
    parents=True,
    exist_ok=True,
)

figure_labels = [
    "Distance: 200 vs 100 km",
    "Freight: 20 vs 10 BRL",
    "Purchase holiday vs non-holiday",
    "Holiday count in estimated window: 1 vs 0",
    "Rainfall index: 5 vs 0 mm",
]

fig_or, ax_or = plt.subplots(
    figsize=(11, 6.5),
    constrained_layout=True,
)

positions = np.arange(len(focus_features))

plot_specs = [
    ("M4_transformed", -0.12, "Without calendar adjustment"),
    ("M4_time_adjusted", 0.12, "With calendar adjustment"),
]

for model_name, offset, label in plot_specs:
    rows = (
        adjusted_odds_ratios.loc[
            adjusted_odds_ratios["model"].eq(model_name)
            & adjusted_odds_ratios["block_days"].eq(7)
            & adjusted_odds_ratios["feature"].isin(focus_features)
        ]
        .set_index("feature")
        .loc[focus_features]
    )

    estimates = rows["adjusted_or"].to_numpy()
    lower = rows["ci_lower"].to_numpy()
    upper = rows["ci_upper"].to_numpy()

    ax_or.errorbar(
        estimates,
        positions + offset,
        xerr=np.vstack([
            estimates - lower,
            upper - estimates,
        ]),
        fmt="o",
        capsize=3,
        label=label,
    )

ax_or.axvline(1, color="gray", linestyle="--")
ax_or.set_xscale("log")
ax_or.set_yticks(positions)
ax_or.set_yticklabels(figure_labels)
ax_or.invert_yaxis()

ax_or.set_xlabel(
    "Adjusted odds ratio (log scale)\n"
    "Pointwise 95% CIs; 65 seven-day clusters\n"
    "Common sample: 56,553 orders; 3,649 late orders"
)
ax_or.set_title(
    "Key M4 associations and calendar-adjustment sensitivity"
)
ax_or.legend(loc="best")
ax_or.grid(axis="x", alpha=0.25)

or_figure_path = (
    regression_figures_dir / "adjusted_or_key_contrasts.png"
)

fig_or.savefig(
    or_figure_path,
    dpi=300,
    bbox_inches="tight",
    facecolor="white",
)

plt.show()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_15068\4023354946.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
residual_figure_path = (
    regression_figures_dir
    / "functional_form_residual_comparison.png"
)

fig_residual_comparison.savefig(
    residual_figure_path,
    dpi=200,
    bbox_inches="tight",
    facecolor="white",
)

exported_paths = [
    sample_path,
    adjusted_or_path,
    diagnostics_path,
    or_figure_path,
    residual_figure_path,
]

print("EXPORTED FILES")

for path in exported_paths:
    assert path.is_file()
    assert path.stat().st_size > 0
    print(
        path.relative_to(PROJECT_ROOT),
        f"({path.stat().st_size:,} bytes)",
    )

print("\nPASS: Regression tables and figures exported.")

EXPORTED FILES
outputs\tables\regression\sample_summary.csv (444 bytes)
outputs\tables\regression\adjusted_odds_ratios.csv (62,587 bytes)
outputs\tables\regression\diagnostics.csv (40,397 bytes)
outputs\figures\regression\adjusted_or_key_contrasts.png (183,773 bytes)
outputs\figures\regression\functional_form_residual_comparison.png (796,134 bytes)

PASS: Regression tables and figures exported.


In [33]:
expected_source_sha256 = (
    "ed604fa22a2c95695036966f80a0f8a4666fb242bc2ab9f781ed8fb68630ff4c"
)

with DATA_PATH.open("rb") as source_file:
    final_source_sha256 = hashlib.file_digest(
        source_file,
        "sha256",
    ).hexdigest()

assert final_source_sha256 == expected_source_sha256
assert final_source_sha256 == dataset_sha256

reloaded_or = pd.read_csv(
    adjusted_or_path,
    encoding="utf-8-sig",
)

assert len(reloaded_or) == 316

for column in ["model", "feature", "comparison", "block_days"]:
    assert (
        reloaded_or[column].astype("string").tolist()
        == adjusted_odds_ratios[column].astype("string").tolist()
    )

numeric_export_columns = [
    "adjusted_or",
    "ci_lower",
    "ci_upper",
    "p_value",
    "log_odds_ratio",
    "standard_error_log_or",
    "orders",
    "late_orders",
    "clusters",
    "inference_df",
    "group_orders",
    "group_late_orders",
]

for column in numeric_export_columns:
    np.testing.assert_allclose(
        reloaded_or[column].to_numpy(dtype=float),
        adjusted_odds_ratios[column].to_numpy(dtype=float),
        rtol=1e-10,
        atol=1e-12,
        equal_nan=True,
    )

reloaded_sample = pd.read_csv(
    sample_path,
    encoding="utf-8-sig",
)

assert len(reloaded_sample) == 3
assert reloaded_sample["dataset_sha256"].eq(
    expected_source_sha256
).all()

for column in ["orders", "not_late_orders", "late_orders"]:
    np.testing.assert_array_equal(
        reloaded_sample[column].to_numpy(),
        sample_summary[column].to_numpy(),
    )

reloaded_diagnostics = pd.read_csv(
    diagnostics_path,
    encoding="utf-8-sig",
    dtype="string",
    keep_default_na=False,
)

pd.testing.assert_frame_equal(
    reloaded_diagnostics,
    diagnostics.astype("string").fillna(""),
    check_dtype=False,
)

for path in [or_figure_path, residual_figure_path]:
    with path.open("rb") as image_file:
        assert image_file.read(8) == b"\x89PNG\r\n\x1a\n"

findings_path = (
    PROJECT_ROOT / "docs" / "explanatory_regression/logistic_regression_findings.md"
)

assert findings_path.is_file()
assert findings_path.stat().st_size > 0

print("Source dataset SHA256:", final_source_sha256)
print("Source dataset matches the original version: PASS")
print("OR table reload: PASS")
print("Sample summary reload: PASS")
print("Diagnostics reload: PASS")
print("PNG signatures: PASS")
print("Findings document exists: PASS")
print("\nFinal regression output verification: PASS")

Source dataset SHA256: ed604fa22a2c95695036966f80a0f8a4666fb242bc2ab9f781ed8fb68630ff4c
Source dataset matches the original version: PASS
OR table reload: PASS
Sample summary reload: PASS
Diagnostics reload: PASS
PNG signatures: PASS
Findings document exists: PASS

Final regression output verification: PASS
